In [1]:
!pip install pyserial


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import serial
import time
import requests
import sys
# LOCKED CONFIGURATION
PORT = 'COM3'
BAUD_RATE = 9600
MessageQueue = []
Ql = 0

In [ ]:
def send(url):

In [ ]:
print(f"--- Hardcoded Serial Listener on {PORT} ---")
print("Press Ctrl+C to terminate the script.\n")

while True:
    try:
        # Open connection directly to COM3
        print(f"Attempting connection to {PORT}...")
        arduino = serial.Serial(port=PORT, baudrate=BAUD_RATE, timeout=1)
        
        # Wait for the Arduino to auto-reboot and settle
        time.sleep(2)
        print(f"Connected to {PORT}! Streaming incoming data now:\n")
        
        # Clear out any stale bootup characters
        arduino.reset_input_buffer()

        # Infinite read loop
        while True:
            if arduino.in_waiting > 0:
                raw_bytes = arduino.readline()
                #print(raw_bytes)
                try:
                    # Clean up trailing newlines and spaces
                    decoded_string = raw_bytes.decode('utf-8').strip()
                    
                    # Print formatted time and the message
                    timestamp = time.strftime("%H:%M:%S")
                    payload = {
                        "timestamp": timestamp,
                        "value": decoded_string
                    }
                    if(len(MessageQueue)>50):
                        print(f"🚀 Queue full! Forwarding all 50 messages to FastAPI...", end="", flush=True)
                        try:
                            print(f" -> Forwarding to FastAPI...", end="", flush=True)
                            response = requests.post("http://127.0.0.1:8000/api/soil/bulk", json=MessageQueue, timeout=5)
                            if response.status_code == 200:
                                print(" Success! ✅", flush=True)
                                MessageQueue.clear()
                            else:
                                print(f" Failed ⚠️ (Status: {response.status_code})", flush=True)
                        except requests.exceptions.ConnectionError:
                            print(" Failed ❌ (FastAPI server unreachable)", flush=True)
                    MessageQueue.append(payload)
                    
                    #print(f"[{timestamp}] {decoded_string}")
                    
                    
                except UnicodeDecodeError:
                    pass # Ignore occasional corrupt bytes during hot-plugs

            # Prevent high CPU utilization
            time.sleep(0.01)

    except serial.SerialException:
        print(f"\n[ERROR] Could not open {PORT}. Is it unplugged or in use by Arduino IDE?")
        print("Reconnecting in 3 seconds...")
        time.sleep(3)

    except KeyboardInterrupt:
        print(f"\nClosing {PORT} interface. Goodbye!")
        if 'arduino' in locals() and arduino.is_open:
            arduino.close()
        sys.exit(0)


--- Hardcoded Serial Listener on COM3 ---
Press Ctrl+C to terminate the script.

Attempting connection to COM3...

[ERROR] Could not open COM3. Is it unplugged or in use by Arduino IDE?
Reconnecting in 3 seconds...
Attempting connection to COM3...

[ERROR] Could not open COM3. Is it unplugged or in use by Arduino IDE?
Reconnecting in 3 seconds...
Attempting connection to COM3...

[ERROR] Could not open COM3. Is it unplugged or in use by Arduino IDE?
Reconnecting in 3 seconds...
Attempting connection to COM3...

[ERROR] Could not open COM3. Is it unplugged or in use by Arduino IDE?
Reconnecting in 3 seconds...
Attempting connection to COM3...

[ERROR] Could not open COM3. Is it unplugged or in use by Arduino IDE?
Reconnecting in 3 seconds...
Attempting connection to COM3...

[ERROR] Could not open COM3. Is it unplugged or in use by Arduino IDE?
Reconnecting in 3 seconds...
Attempting connection to COM3...

[ERROR] Could not open COM3. Is it unplugged or in use by Arduino IDE?
Reconnecti